In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("../data/sales_with_supply_data.csv")
print(df.shape)

features = [
    "sales_units", "holiday_season", "promotion_applied",
    "competitor_price_index", "economic_index", "weather_impact",
    "price", "discount_percentage",
    "region_Europe", "region_North America",
    "store_type_Retail", "store_type_Wholesale",
    "category_Cabinets", "category_Chairs", "category_Sofas", "category_Tables",
]

X = df[features]
y = df["future_demand"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

naive_pred = [y_train.mean()] * len(y_test)
naive_mae = mean_absolute_error(y_test, naive_pred)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)
model_mae = mean_absolute_error(y_test, model_pred)

print("Lazy guess error:", round(naive_mae, 2))
print("Smart model error:", round(model_mae, 2))
print("The model is", round((1 - model_mae/naive_mae)*100, 1), "% better than guessing the average")

(4999, 26)
Training rows: 3999
Testing rows: 1000
Lazy guess error: 48.12
Smart model error: 48.4
The model is -0.6 % better than guessing the average


In [3]:
correlations = df[features + ["future_demand"]].corr()["future_demand"].sort_values(ascending=False)
print(correlations)

future_demand             1.000000
category_Chairs           0.022627
discount_percentage       0.018068
region_Europe             0.017580
promotion_applied         0.016861
weather_impact            0.015891
store_type_Retail         0.013399
category_Cabinets         0.013298
store_type_Wholesale      0.007356
economic_index            0.003462
holiday_season           -0.001009
price                    -0.001106
region_North America     -0.004321
category_Sofas           -0.008348
sales_units              -0.009328
competitor_price_index   -0.011170
category_Tables          -0.017454
Name: future_demand, dtype: float64


In [4]:
daily_sales = df.groupby("date")["sales_units"].sum().reset_index()
daily_sales["date"] = pd.to_datetime(daily_sales["date"])
daily_sales.columns = ["ds", "y"]

print(daily_sales.head())
print("Total days:", len(daily_sales))

          ds    y
0 2023-01-01   99
1 2023-01-02   95
2 2023-01-03  101
3 2023-01-04   33
4 2023-01-05   82
Total days: 4999


In [5]:
!pip install prophet

   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.1 MB 3.9 MB/s eta 0:00:03
   --- ------------------------------------ 1.0/12.1 MB 3.1 MB/s eta 0:00:04
   ------ --------------------------------- 1.8/12.1 MB 2.8 MB/s eta 0:00:04
   ------ --------------------------------- 2.1/12.1 MB 3.0 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/12.1 MB 2.8 MB/s eta 0:00:04
   ----------- ---------------------------- 3.4/12.1 MB 2.6 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.1 MB 2.7 MB/s eta 0:00:03
   --------------- ------------------------ 4.7/12.1 MB 2.7 MB/s eta 0:00:03
   ----------------- ---------------------- 5.2/12.1 MB 2.8 MB/s eta 0:00:03
   ----------------- ---------------------- 5.2/12.1 MB 2.8 MB/s eta 0:00:03
   ----------------- ---------------------- 5.2/12.1 MB 2.8 MB/s eta 0:00:03
   ----------------- ---------------------- 5.2/12.1 MB 2.8 MB/s eta 0:00:03
   ---

In [6]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

# Split into train (first 90%) and test (last 10%) by TIME, not randomly
# (forecasting must always test on the future, never on a random shuffle)
split_point = int(len(daily_sales) * 0.9)
train = daily_sales.iloc[:split_point]
test = daily_sales.iloc[split_point:]

# Lazy guess: repeat the last known value
naive_pred = [train["y"].iloc[-1]] * len(test)
naive_mae = mean_absolute_error(test["y"], naive_pred)

# Real forecasting model
model = Prophet(weekly_seasonality=True, yearly_seasonality=True)
model.fit(train)

future = model.make_future_dataframe(periods=len(test))
forecast = model.predict(future)

# Compare only the test period
forecast_test = forecast.set_index("ds").loc[test["ds"]]["yhat"]
model_mae = mean_absolute_error(test["y"], forecast_test)

print("Lazy guess error:", round(naive_mae, 2))
print("Prophet model error:", round(model_mae, 2))
print("The model is", round((1 - model_mae/naive_mae)*100, 1), "% better than guessing the last value")

23:57:09 - cmdstanpy - INFO - Chain [1] start processing
23:57:14 - cmdstanpy - INFO - Chain [1] done processing


Lazy guess error: 57.03
Prophet model error: 47.79
The model is 16.2 % better than guessing the last value


In [7]:
# Save the full forecast (train + test + a bit into the future) for Power BI
forecast_output = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
forecast_output.columns = ["date", "predicted_demand", "lower_bound", "upper_bound"]

# Also tag which rows were actual test data, so Power BI can show actual vs predicted
actuals = daily_sales.copy()
actuals.columns = ["date", "actual_demand"]

forecast_output = forecast_output.merge(actuals, on="date", how="left")

forecast_output.to_csv("../data/forecast_results.csv", index=False)
print("Saved. Rows:", len(forecast_output))
print(forecast_output.tail(10))

Saved. Rows: 4999
           date  predicted_demand  lower_bound  upper_bound  actual_demand
4989 2036-08-29        103.496432    32.398385   169.817093             25
4990 2036-08-30        102.121860    28.801966   174.806514             41
4991 2036-08-31        100.534914    26.525008   174.484988            151
4992 2036-09-01        105.349439    36.295429   177.153398             47
4993 2036-09-02        106.754476    33.805830   174.226143            107
4994 2036-09-03        103.132780    28.863257   171.889619            145
4995 2036-09-04        108.476215    34.075761   182.747279             30
4996 2036-09-05        107.106649    37.062569   179.595713             65
4997 2036-09-06        106.009342    39.099187   173.075110             55
4998 2036-09-07        104.624583    36.160416   172.830718             52


In [8]:
import os
print(os.listdir("../data"))

['.ipynb_checkpoints', 'forecast_results.csv', 'raw_sales.csv.csv', 'sales_with_supply_data.csv', 'supplier_reference.csv']
